# Safe Bayesian Optimization: sequential benchmark

Three policies on the same one-dimensional synthetic problem: **mean-constrained BO** (posterior-mean feasibility), **LCB/UCB-safe BO** (pessimistic confidence-bound feasibility), and **safe uncertainty exploration**. All use the same observed initial seeds, BoTorch GP models and a finite Sobol candidate pool. This is a small illustrative benchmark, not a statistical safety guarantee.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from botorch.acquisition.analytic import PosteriorMean
from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from gpytorch.mlls import ExactMarginalLogLikelihood

from robotorchan.optim import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])
constraints = (ContinuousConstraint(0, 0.0),)
initial_x = torch.tensor([[0.05], [0.10], [0.15]])


In [ ]:
def objective(x):
    return -(x - 0.80).square()


def safety_output(x):
    return x - 0.65 + 0.10 * torch.sin(8 * x)


def fit_model(x, y):
    model = SingleTaskGP(x, y)
    fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood, model))
    model.eval()
    return model


def candidate_pool(seed, n=256):
    unit = torch.quasirandom.SobolEngine(1, scramble=True, seed=seed).draw(n)
    return bounds[0] + (bounds[1] - bounds[0]) * unit


## Three sequential policies

The mean-constrained baseline uses posterior-mean safety only (not ground-truth constraints). The two conservative policies reuse the library's Phase 6 ask/tell workflow. Models are explicitly refitted after each observation. A posterior-safe point can still be unsafe when evaluated against the true constraint.

In [ ]:
def run_policy(policy, seed=11, steps=6):
    torch.manual_seed(seed)
    x = initial_x.clone()
    y = objective(x)
    g = safety_output(x)
    workflow = None
    if policy != "mean_constrained":
        spec = SafetySpecification(
            constraints=constraints,
            initial_X=x,
            observed_constraints=g,
            beta=4.0,
        )
        workflow = SafeBOWorkflow(spec, bounds)
    history = []
    for step in range(steps):
        objective_model = fit_model(x, y)
        safety_model = fit_model(x, g)
        acquisition = PosteriorMean(objective_model)
        if policy == "mean_constrained":
            choices = candidate_pool(seed + step)
            with torch.no_grad():
                predicted_safe = safety_model.posterior(choices.unsqueeze(-2)).mean[..., 0, 0] <= 0
                if not bool(predicted_safe.any()):
                    break
                values = acquisition(choices[predicted_safe].unsqueeze(-2))
                next_x = choices[predicted_safe][values.argmax()].reshape(1, 1)
        else:
            mode = "objective" if policy == "confidence_safe" else "expansion"
            try:
                next_x = workflow.ask(
                    safety_model,
                    mode=mode,
                    acq_function=acquisition if mode == "objective" else None,
                    num_samples=256,
                    seed=seed + step,
                )
            except RuntimeError as exc:
                if "No posterior-safe candidates" not in str(exc):
                    raise
                break
        next_y, next_g = objective(next_x), safety_output(next_x)
        history.append({
            "step": step + 1,
            "x": next_x.item(),
            "objective": next_y.item(),
            "unsafe": bool((next_g > 0).any()),
        })
        if workflow is not None:
            workflow.tell(next_x, next_g)
        x = torch.cat([x, next_x], dim=0)
        y = torch.cat([y, next_y], dim=0)
        g = torch.cat([g, next_g], dim=0)
    return history


policies = ["mean_constrained", "confidence_safe", "safe_expansion"]
runs = {policy: run_policy(policy) for policy in policies}
for policy, history in runs.items():
    print(policy, "evaluations:", len(history),
          "unsafe:", sum(item["unsafe"] for item in history),
          "best feasible:", max(
              (item["objective"] for item in history if not item["unsafe"]),
              default=float("nan"),
          ))


In [ ]:
fig, ax = plt.subplots()
for policy, history in runs.items():
    ax.plot(
        [item["step"] for item in history],
        [item["x"] for item in history],
        marker="o",
        label=policy,
    )
    unsafe = [item for item in history if item["unsafe"]]
    ax.scatter(
        [item["step"] for item in unsafe],
        [item["x"] for item in unsafe],
        marker="x",
        s=90,
        color="red",
    )
ax.set(xlabel="Iteration", ylabel="Proposed x", title="Candidate trajectories (red x: unsafe)")
ax.legend()
plt.show()


## Interpretation and limitations

Compare the number of truly unsafe evaluations and the best feasible objective, not only candidate trajectories. The mean-constrained baseline may propose points that violate the true constraint; conservative policies may be overly cautious or find no candidates. The expansion policy maximizes safety-model uncertainty **within** the predicted-safe set; it does not implement formal SafeOpt reachability. Results depend on seed, kernel fitting and finite candidate pools. Increase repetitions and report variance before drawing performance conclusions.